### This is Auto Preprocessor for any kind of CSV  data  
##### It uses python basic libraries and RAG pipeline to automate the preproceesing step for machine learning to an extent


In [ ]:
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import chromadb

In [ ]:
DATASET = input("Enter the CSV path: ").strip()
SIMILARITY_THRESHOLD = 0.5
DEFAULT_ENCODING = "onehot"
data = pd.read_csv(DATASET)

In [ ]:
###output cols
mean_imputer = []
median_imputer = []
mode_imputer = []
label_encoder = []
onehot_encoder = []
ordinal_encoder =[]
standard_scaler = []
robust_scaler = []
no_scaling = []
drop_cols = []

In [ ]:

model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(
    path=r"..\AutoPreprocessor\chroma_db"
)
collection = client.get_collection("rag_collection")


In [ ]:
for col in tqdm(data.columns, desc="Classifying Columns"):
    if data[col].isnull().mean() > 0.5:
        drop_cols.append(col)
        continue
    if data[col].dtype == object:
        if data[col].isnull().sum() > 0:
            mode_imputer.append(col)
        if data[col].nunique() == 2:            
            label_encoder.append(col)
        else:
            onehot_encoder.append(col)
    else:
        if data[col].isnull().sum() > 0:
            if abs(data[col].skew()) > 1:
                median_imputer.append(col)
            else:
                mean_imputer.append(col)
        if data[col].nunique()/len(data) < 0.05:
            no_scaling.append(col)
        elif abs(data[col].skew()) > 1:
            robust_scaler.append(col)
        else:
            standard_scaler.append(col)


In [ ]:
for j in tqdm(onehot_encoder, desc="RAG Encoding Prediction"):
    values = pd.Series(data[j].dropna().unique()).astype(str).str.lower().tolist()
    query = " ".join(values)
    query_embedding = model.encode(query)
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=3
    )
    r_documents = results["documents"][0]
    r_distances = results["distances"][0]
    r_metadatas = results["metadatas"][0]
    r_ids = results["ids"][0]
    predictions = []

    for i, (doc, distance, meta, rid) in enumerate(zip(r_documents, r_distances, r_metadatas, r_ids)):
        similarity_score = 1 - distance
        prediction = meta.get("encoding", DEFAULT_ENCODING) if similarity_score > SIMILARITY_THRESHOLD else DEFAULT_ENCODING

        predictions.append({
            "rank": i + 1,
            "id": rid,
            "similarity_score": similarity_score,
            "metadata": meta,
            "document": doc,
            "prediction": prediction
        })
    best = max(predictions, key=lambda x: x["similarity_score"])
    if best["prediction"] == "ordinal":
        ordinal_encoder.append(j)
onehot_encoder = [col for col in onehot_encoder if col not in ordinal_encoder]

In [ ]:

all_lists = [
    mean_imputer,
    median_imputer,
    mode_imputer,
    label_encoder,
    onehot_encoder,
    standard_scaler,
    robust_scaler,
    no_scaling,
    ordinal_encoder
]
for lst in tqdm(all_lists, desc="Cleaning Drop Columns"):
    lst[:] = [col for col in lst if col not in drop_cols]


In [ ]:
print("Mean Imputer:", mean_imputer)
print("Median Imputer:", median_imputer)
print("Mode Imputer:", mode_imputer)
print("Label Encoder:", label_encoder)
print("OneHot Encoder:", onehot_encoder)
print("Ordinal Encoder:", ordinal_encoder)
print("Standard Scaler:", standard_scaler)
print("Robust Scaler:", robust_scaler)
print("No Scaling Required:", no_scaling)
print("Drop Columns:", drop_cols)